# TIES-Merging — pipeline v6 (dataset + grid search di *task-arithmetic*, multi-GPU)

Questo notebook **riscrive** `ties-merging-fast` riprendendo da **task-arithmetic**:

- **Dataset**: un `DegradedImageDataset` per task letto dalle **cartelle**
  `{eval_path}/input/npy` e `{eval_path}/target/npy` (le stesse `dataset_merged/SU|IR|SR`
  di task-arithmetic), con `subsample_pct` per usare solo una fetta di ciascun task.
  *Niente più* validation split dai manifest.
- **Grid search a UN solo stadio** con metrica **normalizzata rispetto agli esperti**
  `score = media_t( MSE_merge[t] / MSE_expert[t] )` (più basso = meglio), baseline
  esperti calcolate una volta sola. *Niente più* proxy v-loss + retention a 2 stadi.

**Cosa resta di TIES:** l'operazione di merge (task_vector → pesi per-task → trim
top-k% → sign election → combine con `merge_method`) e i suoi iperparametri. La
**griglia** è quindi sui knob di TIES (`merge_method`, `keep_ratio`, `lambda_scale`,
`task_lambdas`), ma **ogni combinazione è valutata con la metrica/single-stage di
task-arithmetic**. Questa è l'unica scelta di design non banale: un merge TIES ha
bisogno di `keep_ratio`/`merge_method`, che il grid puramente-lambda di TA non prevede
— così tieni i due elementi richiesti (dataset + stile di ricerca di TA) senza perdere
il merge TIES. Per tornare al TIES classico basta mettere `merge_method_grid=["mean"]`.

**Multi-GPU (2 GPU):** come in task-arithmetic, il codice diventa uno script lanciato
con `accelerate launch --num_processes=2`. Il parallelismo è *data-parallel*: il merge
è deterministico e identico su ogni processo, i `DataLoader` sono shardati da accelerate
e le metriche ricomposte con `gather_for_metrics`.

**Struttura delle celle** (stessa separazione logica del file TIES, una cella = un modulo):
1 setup · 2 **architettura** · 3 **config** · 4 **merge** · 5 **eval** · 6 **grid+main** ·
7 `accelerate launch` · 8 **visualizzazione**. Le celle 2–6 scrivono i moduli in
`/kaggle/working/ties_merge/`; la 7 li esegue su 2 GPU; la 8 gira in-notebook sul best salvato.


In [1]:
# ==============================================================================
# CELLA 1 - SETUP (installa accelerate e crea il package dei moduli)
# ==============================================================================
!pip install -q accelerate
!mkdir -p /kaggle/working/ties_merge

In [2]:
%%writefile /kaggle/working/ties_merge/architecture.py
"""
Architettura del DDPM condizionato (identica al notebook di training e a quella
usata sia in task-arithmetic sia in ties-merging: stessa UNet, stesse chiavi/shape).

Rispetto alla vecchia CELLA 1 di TIES qui e' incluso anche `sample_ddim` come
metodo del wrapper DDPM, perche' la valutazione della grid search (portata da
task-arithmetic) campiona con DDIM tramite `unwrapped.sample_ddim(...)`.
"""

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================================
# ARCHITETTURA (invariata)
# =========================================================================

class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=2, batch_first=True)
        self.ln = nn.LayerNorm(channels)
        self.ff_self = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        size = x.shape[-1]
        x_flat = x.reshape(-1, self.channels, size * size).transpose(1, 2)
        x_ln = self.ln(x_flat)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x_flat
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.transpose(1, 2).reshape(-1, self.channels, size, size)


class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=32, num_groups=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.act1 = nn.SiLU()

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act2 = nn.SiLU()

        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        time_proj = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_proj
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.residual_conv(x)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=3, cond_channels=3, base_channels=64):
        super().__init__()

        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        c = base_channels

        self.inc = UNetBlock(in_channels + cond_channels, c, time_dim)

        self.down1 = nn.Conv2d(c, c, kernel_size=4, stride=2, padding=1)
        self.enc1 = UNetBlock(c, c * 2, time_dim)

        self.down2 = nn.Conv2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = UNetBlock(c * 2, c * 4, time_dim)

        self.down3 = nn.Conv2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = UNetBlock(c * 4, c * 8, time_dim)
        self.attn3 = SelfAttention(c * 8)

        self.down4 = nn.Conv2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)

        self.bottleneck1 = UNetBlock(c * 8, c * 8, time_dim)
        self.attention = SelfAttention(c * 8)
        self.bottleneck2 = UNetBlock(c * 8, c * 8, time_dim)

        self.up1 = nn.ConvTranspose2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.dec1 = UNetBlock(c * 16, c * 4, time_dim)
        self.attn_up1 = SelfAttention(c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.dec2 = UNetBlock(c * 8, c * 2, time_dim)

        self.up3 = nn.ConvTranspose2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = UNetBlock(c * 4, c, time_dim)

        self.up4 = nn.ConvTranspose2d(c, c, kernel_size=4, stride=2, padding=1)
        self.dec4 = UNetBlock(c * 2, c, time_dim)

        self.out = nn.Conv2d(c, in_channels, kernel_size=3, padding=1)

    def forward(self, x_t, t, condition):
        x_input = torch.cat([x_t, condition], dim=1)
        t_emb = self.time_mlp(t.float())

        s1 = self.inc(x_input, t_emb)
        h = self.down1(s1)

        s2 = self.enc1(h, t_emb)
        h = self.down2(s2)

        s3 = self.enc2(h, t_emb)
        h = self.down3(s3)

        s4 = self.enc3(h, t_emb)
        s4 = self.attn3(s4)
        h = self.down4(s4)

        h = self.bottleneck1(h, t_emb)
        h = self.attention(h)
        h = self.bottleneck2(h, t_emb)

        h = self.up1(h)
        h = torch.cat([h, s4], dim=1)
        h = self.dec1(h, t_emb)
        h = self.attn_up1(h)

        h = self.up2(h)
        h = torch.cat([h, s3], dim=1)
        h = self.dec2(h, t_emb)

        h = self.up3(h)
        h = torch.cat([h, s2], dim=1)
        h = self.dec3(h, t_emb)

        h = self.up4(h)
        h = torch.cat([h, s1], dim=1)
        h = self.dec4(h, t_emb)

        return self.out(h)


def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-4, 0.9999)


class DDPMvPrediction(nn.Module):
    def __init__(self, network, n_steps=200, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.network = network
        self.n_steps = n_steps

        beta = cosine_beta_schedule(n_steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])

        sqrt_alpha_bar = torch.sqrt(alpha_bar)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)

        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)
        self.register_buffer('sqrt_alpha_bar', sqrt_alpha_bar)
        self.register_buffer('sqrt_one_minus_alpha_bar', sqrt_one_minus_alpha_bar)
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('posterior_variance', beta * (1 - alpha_bar_prev) / (1 - alpha_bar))

    def forward_diffusion(self, x_0, t, noise):
        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        return sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise

    def forward(self, x_0, condition):
        batch_size = x_0.shape[0]
        t = torch.randint(0, self.n_steps, (batch_size,), device=x_0.device)
        noise = torch.randn_like(x_0)

        x_t = self.forward_diffusion(x_0, t, noise)

        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)

        v_target = sqrt_alpha_bar_t * noise - sqrt_one_minus_alpha_bar_t * x_0
        predicted_v = self.network(x_t, t.float(), condition)

        loss = F.mse_loss(predicted_v, v_target)
        return loss

    @torch.no_grad()
    def sample(self, condition):
        """Sampling DDPM completo (ancestral)."""
        self.network.eval()
        device = self.beta.device

        batch_size, channels, height, width = condition.shape
        x = torch.randn(batch_size, channels, height, width, device=device)

        for i in reversed(range(self.n_steps)):
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_v = self.network(x, t.float(), condition)

            sqrt_alpha_bar_t = self.sqrt_alpha_bar[i]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[i]

            pred_x0 = (sqrt_alpha_bar_t * x) - (sqrt_one_minus_alpha_bar_t * predicted_v)
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            beta_t = self.beta[i]
            alpha_bar_prev_t = self.alpha_bar_prev[i]
            alpha_bar_t = self.alpha_bar[i]
            alpha_t = self.alpha[i]

            mean = (beta_t * torch.sqrt(alpha_bar_prev_t) / (1.0 - alpha_bar_t)) * pred_x0 + \
                   ((1.0 - alpha_bar_prev_t) * torch.sqrt(alpha_t) / (1.0 - alpha_bar_t)) * x

            if i > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(self.posterior_variance[i])
                x = mean + sigma_t * noise
            else:
                x = mean

        self.network.train()
        return x

    @torch.no_grad()
    def sample_ddim(self, condition, ddim_steps=25, eta=0.0):
        """Sampling DDIM veloce e (con eta=0) deterministico dato il rumore iniziale."""
        self.network.eval()
        device = self.beta.device
        batch_size, channels, h, w = condition.shape
        x = torch.randn(batch_size, channels, h, w, device=device)

        step_indices = torch.linspace(0, self.n_steps - 1, ddim_steps, dtype=torch.long)
        step_indices = torch.unique(step_indices).flip(0)

        for idx in range(len(step_indices)):
            i = step_indices[idx].item()
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_v = self.network(x, t.float(), condition)

            alpha_bar_t = self.alpha_bar[i]
            sqrt_alpha_bar_t = self.sqrt_alpha_bar[i]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[i]

            pred_x0 = sqrt_alpha_bar_t * x - sqrt_one_minus_alpha_bar_t * predicted_v
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            if idx == len(step_indices) - 1:
                x = pred_x0
                break

            i_prev = step_indices[idx + 1].item()
            alpha_bar_prev = self.alpha_bar[i_prev]

            pred_noise = (x - sqrt_alpha_bar_t * pred_x0) / (sqrt_one_minus_alpha_bar_t + 1e-8)

            if eta > 0.0:
                sigma_t = eta * torch.sqrt(
                    (1 - alpha_bar_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_prev)
                )
                noise = torch.randn_like(x)
            else:
                sigma_t = torch.tensor(0.0, device=device)
                noise = torch.zeros_like(x)

            dir_xt = torch.sqrt(torch.clamp(1 - alpha_bar_prev - sigma_t ** 2, min=0.0)) * pred_noise
            x = torch.sqrt(alpha_bar_prev) * pred_x0 + dir_xt + sigma_t * noise

        self.network.train()
        return x


def build_model(n_steps=200, base_channels=64, in_channels=3, cond_channels=3, device="cuda"):
    """Factory di comodo: istanzia UNet + wrapper DDPM con gli stessi iperparametri
    del training, cosi' da poter caricare i checkpoint."""
    net = ImprovedUNet(in_channels=in_channels, cond_channels=cond_channels, base_channels=base_channels)
    model = DDPMvPrediction(net, n_steps=n_steps)
    return model.to(device)

Writing /kaggle/working/ties_merge/architecture.py


In [3]:
%%writefile /kaggle/working/ties_merge/configuration.py
# ==============================================================================
# CONFIGURAZIONE AGGIORNATA (Grid Search a 2 Stadi)
# ==============================================================================

CONFIG = {
    # ----- checkpoint di base -----
    "theta0_path": "/kaggle/input/datasets/ilariadarchivio/theta0-pretrain/theta0.pth",
    "ckpt_key": "ema_model",

    # STADIO 1: Task singoli ed Esperti per calcolare le baseline e trovare la "Zona Sicura"
    "tasks_stage1": {
        "star_removal": {
            "expert_ckpt": "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100.pth",
            "eval_path":   "/kaggle/input/datasets/phantasm04/merged/dataset_merged/SU",
        },
        "image_restoration": {
            "expert_ckpt": "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100_IR.pth",
            "eval_path":   "/kaggle/input/datasets/phantasm04/merged/dataset_merged/IR",
        },
        "super_resolution": {
            "expert_ckpt": "/kaggle/input/datasets/ilariadarchivio/epoch-100/ddpmvpred_epoch_100_SR.pth",
            "eval_path":   "/kaggle/input/datasets/phantasm04/merged/dataset_merged/SR",
        },
    },

    # STADIO 2: Cartelle multi-problema per trovare il bilanciamento perfetto
    "tasks_stage2": {
        "IR_SU":    {"eval_path": "/kaggle/input/datasets/phantasm04/merged/dataset_merged/IR_SU"},
        "SR_SU":    {"eval_path": "/kaggle/input/datasets/phantasm04/merged/dataset_merged/SR_SU"},
        "SR_IR":    {"eval_path": "/kaggle/input/datasets/phantasm04/merged/dataset_merged/SR_IR"},
        "SR_IR_SU": {"eval_path": "/kaggle/input/datasets/phantasm04/merged/dataset_merged/SR_IR_SU"},
    },

    # ----- architettura / inferenza -----
    "in_channels": 3,
    "cond_channels": 3,
    "base_channels": 64,
    "n_steps": 200,

    # Percentuale dei dati usata PER OGNI cartella (0.10 = 10%)
    "subsample_pct": 0.10,

    # ==========================================================================
    # GRIGLIA DI PARTENZA TIES (Valutata nello Stadio 1)
    # ==========================================================================
    "merge_method_grid": ["sum"],                 
    "keep_ratio_grid":   [0.2, 0.5, 0.8],
    "lambda_scale_grid": [0.5, 0.8, 1.0, 1.3, 1.6],
    "task_lambdas_grid": [
        None,
        # (1.0, 1.0, 0.5), 
    ],

    # ==========================================================================
    # IMPOSTAZIONI FILTRO "ZONA SICURA" (Da Stadio 1 a Stadio 2)
    # ==========================================================================
    # Massimo rapporto MSE_merge / MSE_esperto tollerato su un singolo task nello Stadio 1.
    # Serve a tagliare fuori i pesi palesemente distruttivi (es: > 2.5 indica prestazioni degenerate).
    "stage1_max_task_ratio": 2.5, 
    
    # Numero massimo di configurazioni promosse allo Stadio 2 (le migliori ordinate per score)
    "stage1_top_k_safe": 15,

    # ----- sampling in valutazione (DDIM veloce) -----
    "use_ddim_for_manual_tests": True,
    "manual_test_ddim_steps": 25,
    "manual_test_ddim_eta": 0.0,

    "eval_batch_size": 8,
    "eval_max_samples_per_task": None,

    "compute_psnr": True,
    "compute_ssim": True,
    "eval_seed_base": 42,

    # ----- output -----
    "output_path": "/kaggle/working/ties.pth",
    "results_json_path": "/kaggle/working/ties_grid_search_results.json",
    "grid_search_top_k": 15,
}

Writing /kaggle/working/ties_merge/configuration.py


In [4]:
%%writefile /kaggle/working/ties_merge/merging.py
# ==============================================================================
# FUNZIONI DI MERGING (solo definizioni: questo modulo non esegue nulla)
# ==============================================================================
# Pipeline TIES:  task_vector = specialista - theta0
#                 -> (pesi per-task) -> trim top-k% -> sign election -> merge
#                 -> theta0 + lambda_scale * merged_tv
#
# Caricamento checkpoint / compatibilita' / task vector: ripresi da task-arithmetic.
# Trim + sign election + merge_method: ripresi da TIES.

from collections import OrderedDict

import torch


# -------------------------------------------------------------------------
# Caricamento / compatibilita' / task vector  (da task-arithmetic)
# -------------------------------------------------------------------------

def load_unet_state_dict(path: str, key: str = None):
    """Carica lo state_dict della UNet, gestendo la chiave (es. 'ema_model') e i
    prefissi '_orig_mod.' / 'network.' dei checkpoint di training."""
    obj = torch.load(path, map_location="cpu")
    sd = obj[key] if (key is not None and isinstance(obj, dict) and key in obj) else obj

    if any(k.startswith("_orig_mod.") for k in sd.keys()):
        sd = {k[len("_orig_mod."):]: v for k, v in sd.items()}

    if any(k.startswith("network.") for k in sd.keys()):
        sd = {k[len("network."):]: v for k, v in sd.items() if k.startswith("network.")}

    return OrderedDict(sd)


def assert_compatible(reference, checkpoints: dict):
    ref_keys = set(reference.keys())
    for name, sd in checkpoints.items():
        keys = set(sd.keys())
        if keys != ref_keys:
            raise ValueError(
                f"[{name}] chiavi non compatibili con theta0.\n"
                f"  mancanti: {ref_keys - keys}\n  in eccesso: {keys - ref_keys}"
            )
        for k in ref_keys:
            if sd[k].shape != reference[k].shape:
                raise ValueError(f"[{name}] shape mismatch su '{k}': {sd[k].shape} vs {reference[k].shape}")


def compute_task_vector(theta_t, theta_0):
    """tau = theta_t - theta_0, solo sui parametri float (gli altri vengono
    lasciati fuori: verranno copiati da theta0 nel merge finale)."""
    tau = OrderedDict()
    for k, v0 in theta_0.items():
        if torch.is_floating_point(v0):
            tau[k] = theta_t[k].float() - v0.float()
    return tau


def task_vector_norm(tau) -> float:
    return torch.sqrt(sum((v ** 2).sum() for v in tau.values())).item()


# -------------------------------------------------------------------------
# Trim + sign election + merge_method  (da TIES)
# -------------------------------------------------------------------------

@torch.no_grad()
def trim_global(task_vector, keep_ratio: float):
    """Trim TIES: azzera tutto tranne il top-k% dei |delta|, soglia GLOBALE."""
    if keep_ratio >= 1.0:
        return {k: v.clone() for k, v in task_vector.items()}
    flat = torch.cat([v.abs().flatten() for v in task_vector.values()])
    k = max(1, int(round(flat.numel() * keep_ratio)))
    threshold = torch.topk(flat, k, largest=True, sorted=False).values.min()
    return {name: v * (v.abs() >= threshold) for name, v in task_vector.items()}


@torch.no_grad()
def ties_merged_tv(task_vectors, keep_ratio: float, task_lambdas=None, merge_method: str = "mean"):
    """Trim -> sign election -> combine. Restituisce il task vector fuso (NON
    dipende dal lambda_scale globale, che viene applicato dopo). `merge_method`:
      - "mean": media sui task concordi (TIES originale). Diluisce l'ampiezza.
      - "sum":  somma dei task concordi. Preserva l'ampiezza del singolo task
                (utile quando la base theta0 e' debole).
      - "dropout_rescale": come "mean" ma rescala di 1/keep_ratio (recupera il
                valore atteso perso col trimming, stile DARE)."""
    if task_lambdas is not None:
        assert len(task_lambdas) == len(task_vectors)
        pairs = [(tv, w) for tv, w in zip(task_vectors, task_lambdas) if w != 0.0]
        task_vectors = [{k: v * w for k, v in tv.items()} for tv, w in pairs]

    trimmed = [trim_global(tv, keep_ratio) for tv in task_vectors]
    rescale = (1.0 / keep_ratio) if (merge_method == "dropout_rescale" and keep_ratio > 0) else 1.0

    merged = OrderedDict()
    for name in trimmed[0].keys():
        stacked = torch.stack([tv[name] for tv in trimmed], dim=0)
        sign = torch.sign(stacked.sum(dim=0)).unsqueeze(0)
        agree = ((torch.sign(stacked) == sign) & (sign != 0)).float()
        summed = (stacked * agree).sum(dim=0)
        if merge_method == "sum":
            merged[name] = summed
        else:  # "mean" o "dropout_rescale"
            merged[name] = summed / agree.sum(dim=0).clamp(min=1.0) * rescale
    return merged


@torch.no_grad()
def build_merged_state_dict(theta_0, merged_tv, lambda_scale: float):
    """state_dict finale della UNet = theta0 + lambda_scale * merged_tv (sui
    parametri float presenti in merged_tv; gli altri vengono copiati da theta0)."""
    merged = OrderedDict()
    for k, v0 in theta_0.items():
        if k in merged_tv and torch.is_floating_point(v0):
            merged[k] = (v0.float() + lambda_scale * merged_tv[k]).to(v0.dtype)
        else:
            merged[k] = v0.clone()
    return merged

Writing /kaggle/working/ties_merge/merging.py


In [5]:
%%writefile /kaggle/working/ties_merge/evaluation.py
# ==============================================================================
# FUNZIONI DI VALUTAZIONE (solo definizioni: questo modulo non esegue nulla)
# ==============================================================================
# Dataset per-task da cartelle + metriche + evaluate_per_task con seed per-task
# fisso: tutto ripreso dal notebook task-arithmetic (single-stage, normalizzato).

import os
import glob
import math
import json
import random
from collections import OrderedDict

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from accelerate import Accelerator
from accelerate.utils import set_seed

from architecture import ImprovedUNet, DDPMvPrediction
from merging import load_unet_state_dict
from configuration import CONFIG


# =========================================================================
# METRICHE PERCETTIVE LEGGERE (nessuna dipendenza esterna)
# =========================================================================

def _to_01(img):
    return (img.clamp(-1.0, 1.0) + 1.0) * 0.5


def psnr_per_sample(pred, target, eps=1e-10):
    p, t = _to_01(pred), _to_01(target)
    mse01 = F.mse_loss(p, t, reduction='none').mean(dim=[1, 2, 3])
    return 10.0 * torch.log10(1.0 / (mse01 + eps))


def _gaussian_window(window_size, sigma, channels, device, dtype):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w1d = g.unsqueeze(1)
    w2d = (w1d @ w1d.t()).unsqueeze(0).unsqueeze(0)
    return w2d.expand(channels, 1, window_size, window_size).contiguous()


def ssim_per_sample(pred, target, window_size=11, sigma=1.5, data_range=1.0):
    p, t = _to_01(pred), _to_01(target)
    channels = p.shape[1]
    window = _gaussian_window(window_size, sigma, channels, p.device, p.dtype)
    pad = window_size // 2

    mu_p = F.conv2d(p, window, padding=pad, groups=channels)
    mu_t = F.conv2d(t, window, padding=pad, groups=channels)
    mu_p2, mu_t2, mu_pt = mu_p * mu_p, mu_t * mu_t, mu_p * mu_t

    sigma_p2 = F.conv2d(p * p, window, padding=pad, groups=channels) - mu_p2
    sigma_t2 = F.conv2d(t * t, window, padding=pad, groups=channels) - mu_t2
    sigma_pt = F.conv2d(p * t, window, padding=pad, groups=channels) - mu_pt

    c1 = (0.01 * data_range) ** 2
    c2 = (0.03 * data_range) ** 2
    ssim_map = ((2 * mu_pt + c1) * (2 * sigma_pt + c2)) / \
               ((mu_p2 + mu_t2 + c1) * (sigma_p2 + sigma_t2 + c2))
    return ssim_map.mean(dim=[1, 2, 3])


# =========================================================================
# DATASET DI VALUTAZIONE (un dataset per singolo task, da cartelle)
# =========================================================================

class DegradedImageDataset(Dataset):
    def __init__(self, root: str, subsample_pct: float = 0.1,
                 normalize_to_minus1_1: bool = True, seed: int = 42):
        self.normalize = normalize_to_minus1_1
        self.samples = []

        input_dir = os.path.join(root, "input", "npy")
        target_dir = os.path.join(root, "target", "npy")
        if not os.path.exists(input_dir) or not os.path.exists(target_dir):
            raise FileNotFoundError(f"Cartelle input/target mancanti sotto: {root}")

        all_files = sorted(os.path.basename(p) for p in glob.glob(os.path.join(input_dir, "*.npy")))
        for f in all_files:
            in_path = os.path.join(input_dir, f)
            tgt_path = os.path.join(target_dir, f)
            if os.path.exists(tgt_path):
                self.samples.append((in_path, tgt_path))

        if not self.samples:
            raise FileNotFoundError(f"Nessuna coppia .npy trovata in: {root}")

        # Campionamento deterministico (stessa fetta su tutti i processi DDP)
        self.samples.sort()
        rng = random.Random(seed)
        num_samples = max(1, int(len(self.samples) * subsample_pct))
        self.samples = rng.sample(self.samples, num_samples)
        self.samples.sort()  # lettura sequenziale efficiente

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        in_path, tgt_path = self.samples[idx]
        x = torch.from_numpy(np.load(in_path).astype(np.float32))
        y = torch.from_numpy(np.load(tgt_path).astype(np.float32))

        if x.ndim == 3 and x.shape[-1] in (1, 3):
            x = x.permute(2, 0, 1)
        if y.ndim == 3 and y.shape[-1] in (1, 3):
            y = y.permute(2, 0, 1)

        if self.normalize:
            x = (x * 2.0) - 1.0
            y = (y * 2.0) - 1.0
        return x, y


def build_task_loaders(tasks_cfg: dict, subsample_pct: float, batch_size: int, accelerator: Accelerator):
    """Un DataLoader (gia' preparato per accelerate) per ciascun task, in ordine STABILE."""
    prepared = OrderedDict()
    for name in sorted(tasks_cfg.keys()):  # ordine fisso -> seed per-task riproducibile
        root = tasks_cfg[name]["eval_path"]
        ds = DegradedImageDataset(root=root, subsample_pct=subsample_pct)
        accelerator.print(f"  [{name:20s}] {len(ds)} campioni ({subsample_pct*100:.1f}%) da {root}")
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
        prepared[name] = accelerator.prepare(loader)
    return prepared


# =========================================================================
# BUILD MODEL / EVALUATE (per-task, con seed per-task fisso)
# =========================================================================

def build_model_fn(state_dict, config=CONFIG):
    unet = ImprovedUNet(
        in_channels=config["in_channels"],
        cond_channels=config["cond_channels"],
        base_channels=config["base_channels"],
    )
    unet.load_state_dict(state_dict)
    unet.eval()
    ddpm = DDPMvPrediction(unet, n_steps=config["n_steps"])
    ddpm.eval()
    return ddpm


@torch.no_grad()
def evaluate_per_task(model, prepared_loaders, accelerator: Accelerator,
                      config=CONFIG, tasks_subset=None):
    """Ritorna { task_name: {mse, psnr, ssim, n} }.
    Ogni task usa set_seed(eval_seed_base + indice_task_fisso): stesso rumore
    iniziale tra esperto e ogni combinazione di merge -> confronto fair."""
    use_ddim = config["use_ddim_for_manual_tests"]
    ddim_steps = config["manual_test_ddim_steps"]
    ddim_eta = config["manual_test_ddim_eta"]
    max_n = config["eval_max_samples_per_task"]
    seed_base = config["eval_seed_base"]

    unwrapped = accelerator.unwrap_model(model)
    ordered_names = list(prepared_loaders.keys())  # ordine stabile

    breakdown = OrderedDict()
    for task_idx, task_name in enumerate(ordered_names):
        if tasks_subset is not None and task_name not in tasks_subset:
            continue

        # rumore iniziale deterministico e fisso per questo task
        set_seed(seed_base + task_idx)

        loader = prepared_loaders[task_name]
        mses, psnrs, ssims = [], [], []
        n_seen = 0

        for x, y in loader:
            if use_ddim:
                pred = unwrapped.sample_ddim(condition=x, ddim_steps=ddim_steps, eta=ddim_eta)
            else:
                pred = unwrapped.sample(condition=x)

            mse_s = F.mse_loss(pred, y, reduction='none').mean(dim=[1, 2, 3])
            mses.extend(accelerator.gather_for_metrics(mse_s).cpu().tolist())

            if config["compute_psnr"]:
                psnrs.extend(accelerator.gather_for_metrics(psnr_per_sample(pred, y)).cpu().tolist())
            if config["compute_ssim"]:
                ssims.extend(accelerator.gather_for_metrics(ssim_per_sample(pred, y)).cpu().tolist())

            n_seen = len(mses)
            if max_n is not None and n_seen >= max_n:
                break

        avg_mse = sum(mses) / len(mses) if mses else float("nan")
        avg_psnr = (sum(psnrs) / len(psnrs)) if psnrs else None
        avg_ssim = (sum(ssims) / len(ssims)) if ssims else None
        breakdown[task_name] = {"mse": avg_mse, "psnr": avg_psnr, "ssim": avg_ssim, "n": len(mses)}

        msg = f"    [{task_name:20s}] MSE={avg_mse:.6f}"
        if avg_psnr is not None:
            msg += f"  PSNR={avg_psnr:.2f}dB"
        if avg_ssim is not None:
            msg += f"  SSIM={avg_ssim:.4f}"
        msg += f"  (n={len(mses)})"
        accelerator.print(msg)

    return breakdown


def normalized_score(merge_breakdown: dict, expert_mse: dict) -> float:
    """media_t( MSE_merge[t] / MSE_expert[t] ). Invariante alla scala tra task."""
    ratios = []
    for t, stats in merge_breakdown.items():
        base = expert_mse.get(t, None)
        if base and base > 0 and math.isfinite(stats["mse"]):
            ratios.append(stats["mse"] / base)
    return sum(ratios) / len(ratios) if ratios else float("inf")


# =========================================================================
# BASELINE ESPERTI (una volta sola)
# =========================================================================

def compute_expert_baselines(tasks_cfg, ckpt_key, prepared_loaders, accelerator, config=CONFIG):
    accelerator.print("\nBaseline esperti (ognuno valutato SOLO sul proprio task):")
    expert_mse = {}
    for name in sorted(tasks_cfg.keys()):
        sd = load_unet_state_dict(tasks_cfg[name]["expert_ckpt"], key=ckpt_key)
        model = build_model_fn(sd, config=config).to(accelerator.device)
        bd = evaluate_per_task(model, prepared_loaders, accelerator, config=config, tasks_subset={name})
        expert_mse[name] = bd[name]["mse"]

        del model
        accelerator.free_memory()
        torch.cuda.empty_cache()

    accelerator.print(f"  -> MSE esperti: {json.dumps({k: round(v, 6) for k, v in expert_mse.items()})}")
    return expert_mse

Writing /kaggle/working/ties_merge/evaluation.py


In [6]:
%%writefile /kaggle/working/ties_merge/search.py
# ==============================================================================
# GRID SEARCH + MAIN A DUE STADI
# ==============================================================================

import os
import sys
import json
import math
import itertools

import torch
from accelerate import Accelerator

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from configuration import CONFIG
from merging import (
    load_unet_state_dict, assert_compatible, compute_task_vector, task_vector_norm,
    ties_merged_tv, build_merged_state_dict,
)
from evaluation import (
    build_task_loaders, build_model_fn, evaluate_per_task,
    normalized_score, compute_expert_baselines,
)

def generate_merge_grid(config=CONFIG):
    for method in config["merge_method_grid"]:
        for tl in config["task_lambdas_grid"]:
            for keep in config["keep_ratio_grid"]:
                for lam in config["lambda_scale_grid"]:
                    yield (method, tl, keep, lam)

def main(config=CONFIG):
    accelerator = Accelerator()
    
    # L'ordine stabile dei vettori dipende sempre dai 3 esperti fondamentali dello Stadio 1
    tasks_s1_cfg = config["tasks_stage1"]
    ordered_names = sorted(tasks_s1_cfg.keys())  

    # --- 1. Caricamento Theta0 ---
    theta0 = load_unet_state_dict(config["theta0_path"])
    accelerator.print(f"[theta0] Caricato da {config['theta0_path']}")

    # --- 2. Checkpoint Esperti + Calcolo Task Vectors ---
    checkpoints = {
        name: load_unet_state_dict(tasks_s1_cfg[name]["expert_ckpt"], key=config["ckpt_key"])
        for name in tasks_s1_cfg
    }
    assert_compatible(theta0, checkpoints)
    
    task_vectors = {name: compute_task_vector(checkpoints[name], theta0) for name in ordered_names}
    task_vectors_ordered = [task_vectors[name] for name in ordered_names]
    
    accelerator.print("\nNorme dei Task Vector originali:")
    for name in ordered_names:
        accelerator.print(f"  {name:20s}: {task_vector_norm(task_vectors[name]):.4f}")

    # =========================================================================
    # STADIO 1: Valutazione sui singoli task e isolamento Zona Sicura
    # =========================================================================
    accelerator.print("\n" + "="*74)
    accelerator.print("INIZIO STADIO 1: Ricerca Zona Sicura su singoli task")
    accelerator.print("="*74)
    
    prepared_loaders_s1 = build_task_loaders(
        tasks_s1_cfg, config["subsample_pct"], config["eval_batch_size"], accelerator
    )
    
    expert_mse = compute_expert_baselines(
        tasks_s1_cfg, config["ckpt_key"], prepared_loaders_s1, accelerator, config=config
    )

    combos = list(generate_merge_grid(config))
    total_combos = len(combos)
    tv_cache = {}   
    stage1_results = []

    for i, (method, tl, keep, lam) in enumerate(combos, start=1):
        cache_key = (method, tl, keep)
        if cache_key not in tv_cache:
            tv_cache[cache_key] = ties_merged_tv(
                task_vectors_ordered, keep, task_lambdas=tl, merge_method=method
            )
        
        merged_sd = build_merged_state_dict(theta0, tv_cache[cache_key], lam)
        model = build_model_fn(merged_sd, config=config).to(accelerator.device)

        tl_str = "None" if tl is None else "(" + ",".join(f"{w:g}" for w in tl) + ")"
        accelerator.print(f"  -> [{i}/{total_combos}] STADIO 1: method={method} tl={tl_str} keep={keep:.2f} lambda={lam:.2f}")

        breakdown = evaluate_per_task(model, prepared_loaders_s1, accelerator, config=config)
        score = normalized_score(breakdown, expert_mse)
        accelerator.print(f"      Score normalizzato globale: {score:.4f}")

        # Controllo se rompe palesemente anche un singolo task
        is_broken = False
        ratios = {}
        for t, stats in breakdown.items():
            ratio = stats["mse"] / expert_mse[t] if expert_mse.get(t) else 1.0
            ratios[t] = ratio
            if ratio > config["stage1_max_task_ratio"]:
                is_broken = True

        if is_broken:
            accelerator.print(f"      [SCARTATO] Almeno un task supera il rapporto critico di {config['stage1_max_task_ratio']}")

        stage1_results.append({
            "merge_method": method, "task_lambdas": tl, "keep_ratio": keep, "lambda_scale": lam,
            "score": score, "breakdown": breakdown, "ratios": ratios, "is_broken": is_broken
        })

        del model
        accelerator.free_memory()
        torch.cuda.empty_cache()

    # Estrazione Zona Sicura
    safe_combos = [r for r in stage1_results if not r["is_broken"]]
    if not safe_combos:
        accelerator.print("\n[ATTENZIONE] Nessuna configurazione rispetta i limiti rigidi. Forzo l'uso delle migliori in assoluto.")
        safe_combos = stage1_results

    safe_combos_sorted = sorted(safe_combos, key=lambda r: r["score"])
    safe_zone = safe_combos_sorted[:config["stage1_top_k_safe"]]

    accelerator.print(f"\n>>> ZONA SICURA SELEZIONATA: {len(safe_zone)} configurazioni promosse allo Stadio 2.")

    # =========================================================================
    # STADIO 2: Ottimizzazione e bilanciamento perfetto su multi-problema
    # =========================================================================
    accelerator.print("\n" + "="*74)
    accelerator.print("INIZIO STADIO 2: Test di bilanciamento su cartelle multi-problema")
    accelerator.print("="*74)

    tasks_s2_cfg = config["tasks_stage2"]
    prepared_loaders_s2 = build_task_loaders(
        tasks_s2_cfg, config["subsample_pct"], config["eval_batch_size"], accelerator
    )

    stage2_results = []

    for i, r in enumerate(safe_zone, start=1):
        method, tl, keep, lam = r["merge_method"], r["task_lambdas"], r["keep_ratio"], r["lambda_scale"]
        cache_key = (method, tl, keep)
        
        merged_sd = build_merged_state_dict(theta0, tv_cache[cache_key], lam)
        model = build_model_fn(merged_sd, config=config).to(accelerator.device)

        tl_str = "None" if tl is None else "(" + ",".join(f"{w:g}" for w in tl) + ")"
        accelerator.print(f"  -> [{i}/{len(safe_zone)}] STADIO 2: method={method} tl={tl_str} keep={keep:.2f} lambda={lam:.2f}")

        breakdown_s2 = evaluate_per_task(model, prepared_loaders_s2, accelerator, config=config)
        
        # Metrica Stadio 2: dal momento che non ci sono esperti nativi combinati,
        # ottimizziamo per il minimo MSE medio calcolato sulle cartelle multi-problema.
        mses_s2 = [stats["mse"] for stats in breakdown_s2.values() if math.isfinite(stats["mse"])]
        score_s2 = sum(mses_s2) / len(mses_s2) if mses_s2 else float("inf")
        accelerator.print(f"      MSE Medio Combinato Stadio 2: {score_s2:.6f}")

        stage2_results.append({
            "merge_method": method, "task_lambdas": tl, "keep_ratio": keep, "lambda_scale": lam,
            "stage1_score": r["score"], "stage2_score": score_s2,
            "breakdown_stage1": r["breakdown"], "breakdown_stage2": breakdown_s2
        })

        del model
        accelerator.free_memory()
        torch.cuda.empty_cache()

    # Ordinamento finale basato sulle performance dello Stadio 2
    stage2_sorted = sorted(stage2_results, key=lambda r: r["stage2_score"])
    top_k = config["grid_search_top_k"]

    accelerator.print("\n" + "=" * 74)
    accelerator.print(f"CLASSIFICA FINALE (ORDINATA PER PRESTAZIONI MULTI-PROBLEMA STADIO 2):")
    for idx, r in enumerate(stage2_sorted[:top_k], start=1):
        tl_str = "None" if r["task_lambdas"] is None else "(" + ",".join(f"{w:g}" for w in r["task_lambdas"]) + ")"
        accelerator.print(f"  {idx}. MSE MEDIO S2 = {r['stage2_score']:.6f} | (Score S1 = {r['stage1_score']:.4f})")
        accelerator.print(f"     [Config]: method={r['merge_method']} tl={tl_str} keep={r['keep_ratio']:.2f} lambda={r['lambda_scale']:.2f}")
        for folder, stats in r["breakdown_stage2"].items():
            accelerator.print(f"       -> {folder}: MSE={stats['mse']:.6f} | PSNR={stats['psnr']:.2f}dB")
    accelerator.print("=" * 74)

    # --- Salvataggio del modello Vincitore Assoluto ---
    best = stage2_sorted[0]
    accelerator.print(f"\n[WINNER] Configurazione migliore individuata per bilanciamento perfetto:")
    accelerator.print(f"  method={best['merge_method']}, task_lambdas={best['task_lambdas']}, keep={best['keep_ratio']}, lambda={best['lambda_scale']}")
    
    cache_key = (best["merge_method"], best["task_lambdas"], best["keep_ratio"])
    final_tv = tv_cache[cache_key]
    final_state_dict = build_merged_state_dict(theta0, final_tv, best["lambda_scale"])

    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        torch.save(final_state_dict, config["output_path"])
        accelerator.print(f"\nModello ottimizzato a due stadi salvato correttamente in: {config['output_path']}")

        # Salvataggio del log completo JSON
        serializable = [
            {"merge_method": r["merge_method"], "task_lambdas": r["task_lambdas"],
             "keep_ratio": r["keep_ratio"], "lambda_scale": r["lambda_scale"],
             "stage1_score": r["stage1_score"], "stage2_score": r["stage2_score"], 
             "breakdown_stage2": r["breakdown_stage2"]}
            for r in stage2_sorted
        ]
        with open(config["results_json_path"], "w") as fh:
            json.dump({"task_order_s1": ordered_names, "expert_mse_s1": expert_mse, "results": serializable}, fh, indent=2, default=str)
        accelerator.print(f"Storico completo archiviato in: {config['results_json_path']}")

if __name__ == "__main__":
    main()

Writing /kaggle/working/ties_merge/search.py


In [7]:
# ==============================================================================
# CELLA 7 - GRID SEARCH SU 2 GPU (esegue search.py e salva ties.pth)
# ==============================================================================
!accelerate launch --num_processes=2 /kaggle/working/ties_merge/search.py

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[theta0] Caricato da /kaggle/input/datasets/ilariadarchivio/theta0-pretrain/theta0.pth

Norme dei Task Vector originali:
  image_restoration   : 50.3134
  star_removal        : 56.4456
  super_resolution    : 80.9864

INIZIO STADIO 1: Ricerca Zona Sicura su singoli task
  [image_restoration   ] 20 campioni (10.0%) da /kaggle/input/datasets/phantasm04/merged/dataset_merged/IR
  [star_removal        ] 60 campioni (10.0%) da /kaggle/input/datasets/phantasm04/merged/dataset_merged/SU
  [super_resolution    ] 60 campioni (10.0%) da /kaggle/

In [8]:
# ==============================================================================
# CELLA 8 - VISUALIZZAZIONE (input | merge | specialista | target)
# ==============================================================================
# Gira IN-NOTEBOOK su una sola GPU (la grid search a 2 GPU e' gia' finita e ha
# salvato ties.pth). Legge i dati dalla STESSA cartella eval_path del task scelto,
# esattamente come in task-arithmetic. Mostra anche lo specialista come riferimento
# del massimo raggiungibile.
import sys, os
sys.path.insert(0, "/kaggle/working/ties_merge")

import torch
import matplotlib.pyplot as plt

from configuration import CONFIG
from architecture import ImprovedUNet, DDPMvPrediction
from merging import load_unet_state_dict
from evaluation import DegradedImageDataset

# ----- cosa visualizzare -----
VIZ_TASK       = "star_removal"          # "star_removal" | "image_restoration" | "super_resolution"
INDICES        = [10, 20, 30]            # indici nel dataset del task
VIZ_DDIM_STEPS = 50
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# ----- modello fuso (best della grid search) -----
unet = ImprovedUNet(in_channels=CONFIG["in_channels"], cond_channels=CONFIG["cond_channels"],
                    base_channels=CONFIG["base_channels"])
unet.load_state_dict(torch.load(CONFIG["output_path"], map_location=DEVICE))
merged = DDPMvPrediction(unet, n_steps=CONFIG["n_steps"]).to(DEVICE).eval()

# ----- specialista di riferimento per lo stesso task -----
spec_unet = ImprovedUNet(in_channels=CONFIG["in_channels"], cond_channels=CONFIG["cond_channels"],
                         base_channels=CONFIG["base_channels"])
spec_sd = load_unet_state_dict(CONFIG["tasks"][VIZ_TASK]["expert_ckpt"], key=CONFIG["ckpt_key"])
spec_unet.load_state_dict(spec_sd)
specialist = DDPMvPrediction(spec_unet, n_steps=CONFIG["n_steps"]).to(DEVICE).eval()

# ----- dati dal task scelto (dataset completo, poi si prendono gli indici) -----
ds = DegradedImageDataset(root=CONFIG["tasks"][VIZ_TASK]["eval_path"], subsample_pct=1.0)
INDICES = [i for i in INDICES if i < len(ds)]
print(f"Task '{VIZ_TASK}': dataset di {len(ds)} immagini, visualizzo gli indici {INDICES}")

def to_img(t):
    return ((t.clamp(-1, 1) + 1) / 2).cpu().permute(1, 2, 0).numpy()

def psnr(pred, target):
    p = (pred.clamp(-1, 1) + 1) / 2
    g = (target.clamp(-1, 1) + 1) / 2
    return 10 * torch.log10(1.0 / ((p - g) ** 2).mean().clamp(min=1e-10)).item()

n = len(INDICES)
fig, ax = plt.subplots(n, 4, figsize=(16, 4 * n))
if n == 1:
    ax = ax[None, :]

torch.manual_seed(0)  # stesso rumore iniziale DDIM per merge e specialista
with torch.no_grad():
    for row, idx in enumerate(INDICES):
        cond, target = ds[idx]
        cond_b = cond.unsqueeze(0).to(DEVICE)

        g = torch.Generator(device=DEVICE).manual_seed(1000 + idx)
        x_T = torch.randn(cond_b.shape, generator=g, device=DEVICE)

        # riusa lo stesso x_T per un confronto equo
        def sample(model):
            model.network.eval()
            x = x_T.clone()
            steps = torch.linspace(0, model.n_steps - 1, VIZ_DDIM_STEPS, dtype=torch.long)
            steps = torch.unique(steps).flip(0)
            for j in range(len(steps)):
                i = steps[j].item()
                tb = torch.full((1,), i, device=DEVICE, dtype=torch.long)
                v = model.network(x, tb.float(), cond_b)
                sab = model.sqrt_alpha_bar[i]; s1 = model.sqrt_one_minus_alpha_bar[i]
                x0 = (sab * x - s1 * v).clamp(-1, 1)
                if j == len(steps) - 1:
                    x = x0; break
                ip = steps[j + 1].item(); abp = model.alpha_bar[ip]
                eps = (x - sab * x0) / (s1 + 1e-8)
                x = abp.sqrt() * x0 + (1 - abp).sqrt() * eps
            return x[0]

        pred_merge = sample(merged)
        pred_spec = sample(specialist)

        ax[row][0].imshow(to_img(cond));       ax[row][0].set_title(f"input (idx {idx})"); ax[row][0].axis("off")
        ax[row][1].imshow(to_img(pred_merge)); ax[row][1].set_title(f"merge ({psnr(pred_merge, target.to(DEVICE)):.1f} dB)", color="darkgreen"); ax[row][1].axis("off")
        ax[row][2].imshow(to_img(pred_spec));  ax[row][2].set_title(f"specialista ({psnr(pred_spec, target.to(DEVICE)):.1f} dB)"); ax[row][2].axis("off")
        ax[row][3].imshow(to_img(target));     ax[row][3].set_title("target"); ax[row][3].axis("off")

plt.tight_layout(); plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/ties.pth'